# Week 5 extension — generated Python becomes an agent

Keep the Week 3 exercise: ask for code, run it separately, inspect the result. Now return errors and checks to the model so it can choose a revision. The model chooses actions; the surrounding Python validates and executes them.

Our small synthetic CSV contains a group and a measurement, including missing values. The independent reference calculation is kept outside the sandbox. This is a computation exercise, not evidence of scientific significance.

Prepare Docker before class (`docker pull python:3.13-slim`). The live section uses `google-genai`, `GEMINI_API_KEY`, and `GEMINI_MODEL` from the local environment. It is disabled by default. No model-generated Python runs on the host.


In [ ]:
from pathlib import Path
import sys, json
root = Path.cwd() if (Path.cwd() / 'labs').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root / 'labs'))
from course_tools import load_measurements, summarize_measurements, run_agent, gemini_text
from python_sandbox import DockerPythonSandbox, check_analysis_execution

expected = summarize_measurements(load_measurements())  # verifier, not sent to the model
sandbox = DockerPythonSandbox(root / 'data/measurements.csv')
TASK = """Read /input/data.csv using Python's standard library.
Columns: sample, group, value, unit. Empty values are missing, not zero.
For each group print rows, observed, missing, and the observed mean.
Print exactly one JSON object: {group: {rows, observed, missing, mean}}.
Use null for a mean with no observed measurements.
Choose a tool to execute code, inspect feedback, and revise until checks pass.
"""

def execute_python(code):
    execution = sandbox.execute_python(code)
    return {'execution': execution, 'checks': check_analysis_execution(execution, expected)}

def verify_final(final, state):
    checks = [o.get('result', {}).get('checks', {}) for o in state['observations']]
    return final == expected and any(c.get('passed') for c in checks)


## 1) Inspect feedback offline

This scripted demonstration uses simulated tool observations, without calling Docker or a model. It shows an execution error, a revision, and checked completion. The live version below uses the same loop with actual model decisions and container execution.


In [ ]:
def simulated_execution(code):
    if code == 'first proposal':
        execution = {'status': 'completed', 'exit_code': 1, 'output': "ValueError: could not convert empty value"}
    else:
        execution = {'status': 'completed', 'exit_code': 0, 'output': json.dumps(expected)}
    return {'execution': execution, 'checks': check_analysis_execution(execution, expected)}

def scripted_action(state):
    if not state['observations']:
        return {'tool': 'execute_python', 'args': {'code': 'first proposal'}}
    result = state['observations'][-1]['result']
    if result['checks']['passed']:
        return {'final': result['checks']['result']}
    return {'tool': 'execute_python', 'args': {'code': 'revised proposal'}}

simulation = run_agent(TASK, scripted_action, {'execute_python': simulated_execution}, verify_final, max_steps=4)
assert simulation['status'] == 'verified'
for row in simulation['trace']:
    print(json.dumps(row, indent=2))


## 2) Run with real model-selected code

Enable only after Docker and the API environment are ready. `max_steps` counts decisions, including invalid actions and rejected final answers. The container limits runtime, memory, network access, and output size. The API adapter has a timeout; six calls are the maximum decision budget, not a dollar budget.

Inspect the trace: which error caused which change? Did the model finish too soon, ignore missing values, or use a tool that was not allowed? Save any private trace under `local/runs/`.


In [ ]:
RUN_LIVE = False

def model_action(state):
    prompt = (
        'Return exactly one JSON object, without markdown. Choose either '
        '{"tool":"execute_python","args":{"code":"Python source"}} '
        'or {"final": <computed group summary>}. '
        'The execution tool returns errors and independent checks. '
        'Do not declare completion before a successful check.\n'
        + json.dumps(state)
    )
    return gemini_text(prompt)

if RUN_LIVE:
    from dotenv import load_dotenv
    load_dotenv(root / '.env')
    live_run = run_agent(TASK, model_action, {'execute_python': execute_python}, verify_final, max_steps=6)
    print(json.dumps(live_run, indent=2))


## 3) Compare and explain

Run the same task with one generated program and with the loop. Record verified correctness, number of calls, elapsed time, review/repair time, and provider usage/cost. Add an all-missing group or malformed value and state how it should be handled before rerunning. A successful execution is not the same as a correct analysis.

Then compare with [Gemini's managed code execution](https://ai.google.dev/gemini-api/docs/code-execution) or [Claude's code-execution tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool). Supply the same public data through the provider's supported input mechanism; our local container path is not accessible to a remote service. Check what the provider handles: tool dispatch, runtime, files, state, and traces. Keep your own completion checks.

Optional: make retrieval from the Week 5 notebook another allowed tool. Decide when the model should search instead of answering from supplied context. Preserve a simple baseline; more tools do not by themselves improve performance.
